# 06 - Fine-tune the verdict model
The zero-shot NLI model reaches about 59% accuracy on SciFact. Here it is fine-tuned on SciFact's own training claims: gold rationale + claim becomes entailment or contradiction, and top-ranked non-evidence sentences become neutral.

15% of the train claims are held out. They are used for early stopping and for tuning thresholds, so the official validation split is only used for the final comparison. **Use a GPU runtime** (Runtime > Change runtime type > T4). Training takes a few minutes.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from src import train_nli

res = train_nli.train(epochs=3)
res

## Compare against the zero-shot model
Both models are scored the same way: thresholds tuned on the held-out train claims, results reported on validation.

In [ ]:
import pandas as pd
from src import data, evaluate, train_nli
from src.config import ART
from src.retrieval import Retriever
from src.verdict import Verifier

r = Retriever()
train, val = data.claims('train'), data.claims('validation')
dev = train_nli.split_claims(train)[1]
models = {'zero-shot': None, 'fine-tuned': str(ART / 'nli')}
scores = {}
for name, path in models.items():  # no caching, so a retrained model is never scored from stale results
    v = Verifier(r, nli_model=path)
    scores[name] = (evaluate.score_all(v, dev), evaluate.score_all(v, val))
len(dev), len(val)

In [ ]:
rows, best = {}, {}
for name, (sd, sv) in scores.items():
    best[name] = evaluate.tune(sd, dev.label)
    for tag, (ts, tc) in {'thresholds 0.5': (.5, .5), 'tuned on held-out': (best[name]['ts'], best[name]['tc'])}.items():
        rep = evaluate.report(sv, val.label, ts, tc)
        rows[(name, tag)] = {'macro_f1': rep['macro_f1'], 'accuracy': rep['accuracy'], 'SUPPORT recall': rep['per_class']['SUPPORT']['recall']}
pd.DataFrame(rows).T.round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from src.config import LABELS
from src.verdict import decide

sv = scores['fine-tuned'][1]
pred = [decide(a, b, best['fine-tuned']['ts'], best['fine-tuned']['tc']) for a, b in zip(sv.sup, sv.con)]
ConfusionMatrixDisplay.from_predictions(val.label, pred, labels=list(LABELS), xticks_rotation=20)

## Keep it only if it clearly wins
Compare the fine-tuned macro-F1 on validation with the zero-shot row (about 0.585 earlier). A gain of a point or two is within noise on 300 claims. If the gain is clear, save the thresholds for this model and publish it. If not, skip the next two cells and keep the zero-shot model.

In [ ]:
from src.verdict import save_thresholds

b = best['fine-tuned']
save_thresholds(b['ts'], b['tc'], 'Anish-25/fact-and-logic-nli')
print(open('thresholds.json').read())  # copy this into thresholds.json in your repo

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi, login

login(token=getpass('Paste your HF write token: '))
repo = 'Anish-25/fact-and-logic-nli'
api = HfApi()
api.create_repo(repo, exist_ok=True)
api.upload_folder(folder_path=str(ART / 'nli'), repo_id=repo)